In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from utils import load_config
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio
import plotly.graph_objects as go

from sklearn.preprocessing import MinMaxScaler # used to normalize data for radar charts # conda install -c conda-forge scikit-learn   
from scipy.spatial.distance import euclidean
from fastdtw import fastdtw # pip install fastdtw

# To handle curvature issue 
# robot stagnate or slowly moves curvature explode
from scipy.signal import medfilt
from scipy.interpolate import interp1d

**Handling velocity**

In [ ]:
def model_only_segments(df, teleop_col='lin_x_teleop'):
    """Return boolean mask and segment indices for contiguous model-controlled segments."""

    if teleop_col not in df.columns:
        df[teleop_col] = 0.0  # No intervention no teleop column

    mask_model_only = df[teleop_col] == 0.0
    # label contiguous True regions
    # diff on mask: True->False transitions produce boundaries
    idx = np.where(mask_model_only)[0]
    if len(idx) == 0:
        print("No model segments found")
        return []  # no model segments
    # find breaks where indices are not consecutive
    breaks = np.where(np.diff(idx) != 1)[0] # if difference between consecutive indices is not 1, we have a jump in indices meaning we used teleoperation
    segments = []
    start = idx[0]
    for b in breaks:
        end = idx[b]
        segments.append((start, end))
        start = idx[b+1]
    segments.append((start, idx[-1]))
    return segments, breaks

## Process data

In [ ]:
reference_header = 'reference'

config = load_config()
# TODO Handle now that we cannot be inside docker using plotly anymore

df = pd.read_csv(f"../dataframes/all_data_20251029-030058.csv") #Index(['pose_x', 'pose_y', 'goal', 'robot', 'environment', 'env_type','augmentation'],
df.head()

# Compute path lenght

In [ ]:
# Take into account the whole path itself (with intervention)

def compute_path_length(df: pd.DataFrame):
    path_lengths = []
    grouped = df.groupby(['robot', 'environment', 'augmentation', 'env_type'])
    for (robot, environment, augmentation, env_type), group in grouped:
        # print(f"Computing path length for {robot} in {environment} with {augmentation} ({env_type})")
        positions = group[['pose_x', 'pose_y']].to_numpy()
        length = 0.0
        for i in range(1, len(positions)):
            length += euclidean(positions[i-1], positions[i])
        path_lengths.append({
            'robot': robot,
            'environment': environment,
            'augmentation': augmentation,
            'env_type': env_type,
            'path_length': length
        })
    return pd.DataFrame(path_lengths)


In [ ]:
path_lengths_df = compute_path_length(df)
path_lengths_df.head()



In [ ]:
df = df.merge(path_lengths_df, on=['robot', 'environment', 'augmentation', 'env_type'], how='left')
df.head()

In [ ]:
df[df['augmentation'] == reference_header]["path_length"].any()

# Compute number of intervention

In [ ]:
# Failsafe as some bags don't have teleop data because there was no intervention at all
def intervention_count(df: pd.DataFrame, teleop_col: str = 'lin_x_teleop'):
    intervention_counts = []
    grouped = df.groupby(['robot', 'environment', 'augmentation', 'env_type'])
    for (robot, environment, augmentation, env_type), group in grouped:

        if teleop_col not in group.columns:
            intervention_count = 0        
        else:
            _, breaks = model_only_segments(group, teleop_col=teleop_col)
            intervention_count = len(breaks) if len(breaks) > 0 else 0

        intervention_counts.append({
            'robot': robot,
            'environment': environment,
            'augmentation': augmentation,
            'env_type': env_type,
            'intervention_count': intervention_count
        })
    return pd.DataFrame(intervention_counts)

In [ ]:
intervention_counts_df = intervention_count(df, teleop_col='lin_x_teleop')
intervention_counts_df.head()

In [ ]:
df = df.merge(intervention_counts_df, on=['robot', 'environment', 'augmentation', 'env_type'], how='left')
df.head()

# Computing arrival rate based on CARE.  **(FOR NOW NOT A RATE)**

**Arrival rate** is defined as: *# goal reach w or w/o collisions* for our case its intervention

In [ ]:
def count_arrival(df: pd.DataFrame):
    arrivals = []
    grouped = df.groupby(['robot', 'environment', 'augmentation', 'env_type'])
    for (robot, environment, augmentation, env_type), group in grouped:
        goal_reached = group['goal'].any()
        arrivals.append({
            'robot': robot,
            'environment': environment,
            'augmentation': augmentation,
            'env_type': env_type,
            'arrival': int(goal_reached)
        })
    return pd.DataFrame(arrivals)

In [ ]:
count_arrival_df = count_arrival(df)
count_arrival_df.head()

In [ ]:
df = df.merge(count_arrival_df, on=['robot', 'environment', 'augmentation', 'env_type'], how='left')
df.head()

# Compute safe only arrivals

In [ ]:
def count_arrival_safe(df: pd.DataFrame):
    arrivals = []
    grouped = df.groupby(['robot', 'environment', 'augmentation', 'env_type'])
    for (robot, environment, augmentation, env_type), group in grouped:
        goal_reached = group['goal'].any()

        teleop_used = group['lin_x_teleop'].any() if 'lin_x_teleop' in group.columns else False
        if teleop_used:
            goal_reached = False

        arrivals.append({
            'robot': robot,
            'environment': environment,
            'augmentation': augmentation,
            'env_type': env_type,
            'arrival_safe': int(goal_reached)
        })
    return pd.DataFrame(arrivals)

In [ ]:
count_arrival_safe_df = count_arrival_safe(df)
count_arrival_safe_df.head()

In [ ]:
df = df.merge(count_arrival_safe_df, on=['robot', 'environment', 'augmentation', 'env_type'], how='left')
df.head()

___________________

# Count false goal positive 
**When model predict arrrived at goal but far from ground truth**

In [ ]:
def compute_dist_to_goal(df: pd.DataFrame, reference_header: str = 'reference'):
    assert df["augmentation"].str.contains(reference_header).any(), f"Reference path not found in dataframe only {df['augmentation'].unique()}."
    dist_to_goals = []
    grouped = df.groupby(['robot', 'environment', 'augmentation', 'env_type'])
    for (robot, environment, augmentation, env_type), group in grouped:
        # Get reference goal position
        df_reference = df[df['augmentation'] == reference_header]
        ref_group = df_reference[
            (df_reference['robot'] == robot) &
            (df_reference['environment'] == environment) &
            (df_reference['env_type'] == env_type)
        ]
        if ref_group.empty:
            print(f"No reference data for {robot}, {environment}, {env_type}")
            continue
        goal_pos = ref_group[['pose_x', 'pose_y']].iloc[-1].to_numpy()

        # Compute final distance to goal
        final_pos = group[['pose_x', 'pose_y']].iloc[-1].to_numpy()
        dist_to_goal = euclidean(final_pos, goal_pos)

        dist_to_goals.append({
            'robot': robot,
            'environment': environment,
            'augmentation': augmentation,
            'env_type': env_type,
            'dist_to_goal': dist_to_goal
        })
    return pd.DataFrame(dist_to_goals)

In [ ]:
def count_false_goal_positives(df: pd.DataFrame, goal_threshold: float = 0.5):
    false_positives = []
    grouped = df.groupby(['robot', 'environment', 'augmentation', 'env_type'])
    for (robot, environment, augmentation, env_type), group in grouped:

        
        if 'dist_to_goal' not in group.columns:
            dist_goal_df = compute_dist_to_goal(df)
            df = df.merge(dist_goal_df, on=['robot', 'environment', 'augmentation', 'env_type'], how='left')


        goal_reached = group['goal'].any()
        final_distance = group['dist_to_goal'].iloc[-1]

        false_positive = goal_reached and (final_distance > goal_threshold)

        false_positives.append({
            'robot': robot,
            'environment': environment,
            'augmentation': augmentation,
            'env_type': env_type,
            'false_goal_positive': int(false_positive)
        })
    return pd.DataFrame(false_positives)

## Add refence dataframe

In [ ]:
def load_reference_poses(df: pd.DataFrame, folder_path: str, reference_header: str = 'reference'):
    grouped = df.groupby(['robot', 'environment'])
    for (robot, environment), group in grouped:
        ref_dfs = pd.read_csv(f"{folder_path}/{robot}/{environment}/{reference_header}/{robot}_{environment}_{reference_header}_odom.csv")
        ref_dfs = ref_dfs[['pose.pose.position.x', 'pose.pose.position.y']]
        ref_dfs = ref_dfs.rename(columns={'pose.pose.position.x': 'pose_x', 'pose.pose.position.y': 'pose_y'})
        ref_dfs['robot'] = robot
        ref_dfs['environment'] = environment
        ref_dfs['augmentation'] = reference_header
        ref_dfs['env_type'] = group['env_type'].iloc[0] # iloc to retrieve only one value

    return ref_dfs

## Compute dist to goal

In [ ]:
folder_path = "/home/mae/Documents/GIT/Research/SafeGNM/metrics/dataframes"
ref_dfs = load_reference_poses(df, folder_path)
# print(ref_dfs.head(10))

df = pd.concat([df, ref_dfs], ignore_index=True)
df.head()


In [ ]:
df["augmentation"].unique()

In [ ]:
distance_to_goal_df = compute_dist_to_goal(df)
distance_to_goal_df.head()


In [ ]:
df = df.merge(distance_to_goal_df, on=['robot', 'environment', 'augmentation', 'env_type'], how='left')
df.head()

## Compute false goal positive

In [ ]:
false_positive_df = count_false_goal_positives(df, goal_threshold=0.5)
false_positive_df.head()

In [ ]:
df = df.merge(false_positive_df, on=['robot', 'environment', 'augmentation', 'env_type'], how='left')
df.head()

___________________

# Velocities

_____________________________________

In [ ]:
# def compute_velocity(segment_df):
#     avg_velocity = segment_df[['lin_x_odom']].mean().values[0]
#     return avg_velocity

In [ ]:
def compute_velocity(df: pd.DataFrame):    
    results = []   
    
    for (robot, environment, augmentation, env_type), group in df.groupby(['robot', 'environment', 'augmentation', 'env_type']):
        # Get model-only segments
        segments, _ = model_only_segments(group)
        
        if len(segments) == 0:
            print(f"No model-only segments for ({robot}, {environment}, {augmentation}, {env_type})")
            continue
        
        # Process each segment
        for start_idx, end_idx in segments:
            segment_df = group.iloc[start_idx:end_idx + 1].copy()
            if len(segment_df) < 2:
                continue
            
            # Compute metrics
            avg_velocity = segment_df[['lin_x_model']].mean().values[0]

            results.append({
                'robot': robot,
                'environment': environment,
                'augmentation': augmentation,
                'avg_velocity': avg_velocity,
            })

    return pd.DataFrame(results)

In [ ]:
df_velocity = compute_velocity(df)
df_velocity.head()

In [ ]:
df = df.merge(df_velocity, on=['robot', 'environment', 'augmentation'], how='left')
df.head()

# PLOT METRICS

In [ ]:
# color_sequence = {
# "blur": "#3A5CF0", 
# "brightness": "#EB8E75", 
# "fog": "#93EF98",
# "no_augmentation": "#F3C282",
# "rain": "#82F1F5", 
# "rain_drizzle":  "#11BFFE",
# "rain_heavy":  "#7EF9AF",
# "rain_torrential":  '#06D6A0',
# "reference":  "#FF90F9",
# "sunFlare_overlay":  "#EFB6FF",
# "sunFlare_physic":  "#F74269"
# }

color_sequence = {
"blur": "#3A5CF0",
"no_augmentation": "#F3C282",
"rain_torrential":  '#06D6A0',
"reference":  "#FF90F9",
"sunFlare_physic":  "#F74269"
}

# 'rain_torrential', 'sunFlare_physic', 'blur', 'no_augmentation',
#        'reference'

In [ ]:
plot_df = df.copy()
plot_df.fillna(0, inplace=True)

## Polar charts

In [ ]:
import pandas as pd
import plotly.graph_objects as go
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import colorsys

def adjust_color_brightness(hex_color, factor=1.2):
    """Lighten or darken a hex color."""
    hex_color = hex_color.lstrip('#')
    r, g, b = tuple(int(hex_color[i:i+2], 16) for i in (0, 2, 4))
    h, l, s = colorsys.rgb_to_hls(r/255, g/255, b/255)
    l = max(0, min(1, l * factor))
    r, g, b = colorsys.hls_to_rgb(h, l, s)
    return f'#{int(r*255):02x}{int(g*255):02x}{int(b*255):02x}'



def plot_single_radar_chart(
    df, 
    metrics, 
    robot_type, 
    environment_type,
    color_sequence=None,
    is_first_figure=True
):
    """
    Plot a polar (radar) chart where each polygon corresponds to a data augmentation type.
    Each axis corresponds to a normalized metric value.
    """
    
    filtered_df = df[
        (df['robot'] == robot_type) &
        (df['environment'] == environment_type)
    ]
    
    augmentations = filtered_df['augmentation'].unique()
    fig = go.Figure()

    # Choose a color palette if not provided
    if color_sequence is None:
        color_sequence = px.colors.qualitative.Plotly


    for i, augmentation_type in enumerate(augmentations):
        aug_df = filtered_df[filtered_df['augmentation'] == augmentation_type]

        metric_values = [aug_df[m].mean() for m in metrics]
        plot_df = pd.DataFrame({
            'metric': metrics,
            'value': metric_values
        })

        # Close the polygon
        plot_df = pd.concat([plot_df, plot_df.iloc[[0]]])

        if isinstance(color_sequence, dict):
            base_color = color_sequence.get(augmentation_type, '#888888')
        else:
            base_color = color_sequence[i % len(color_sequence)]


        line_color = adjust_color_brightness(base_color, 0.7)
        fill_color = adjust_color_brightness(base_color, 1.4)


        fig.add_trace(go.Scatterpolar(
            r=plot_df['value'],
            theta=plot_df['metric'],
            fill='toself',
            name=augmentation_type,
            line=dict(color=line_color, width=2),
            fillcolor=fill_color,
            opacity=0.5
        ))


    fig.update_layout(
        polar=dict(
            radialaxis=dict(
                range=[0, 1],
                showticklabels=True,
                ticks=''
            )
        ),
        showlegend=True,
    )

    if is_first_figure:

        fig.update_layout(
            legend=dict(
                title="Augmentations",     # optional
                orientation="v",          # 'h' for horizontal, 'v' for vertical
                yanchor="middle",         # anchor position
                y=0.5,                   # vertical position
                xanchor="left",
                x=0.2                     # horizontal position (0 = left, 1 = right)
            )
     )

    fig.show(renderer="notebook")
    return fig


In [ ]:
polar_chart_all = plot_single_radar_chart(df, color_sequence=color_sequence, metrics=[
    'arrival',
    'arrival_safe',
    'false_goal_positive',
    'intervention_count',
], robot_type="bunker", environment_type="nist_corridor")

In [ ]:
def normalize_by_configuration(df):
    """
    Normalize within each robot-environment pair
    (useful when different robots/environments have very different scales)
    """
    df_norm = df.copy()
    
    for (robot, env), group in df.groupby(['robot', 'environment']):
        mask = (df_norm['robot'] == robot) & (df_norm['environment'] == env)
        
        # Normalize distance to goal within this configuration
        
        df_norm.loc[mask, 'dist_to_goal_norm'] = (
            (group['dist_to_goal'] - group['dist_to_goal'].min()) / 
            (group['dist_to_goal'].max() - group['dist_to_goal'].min())
        )
        
        # Normalize path length within this configuration
        
        df_norm.loc[mask, 'path_length_norm'] = (
            (group['path_length'] - group['path_length'].min()) / 
            (group['path_length'].max() - group['path_length'].min())
        )
        
        # Normalize intervention count
        
        df_norm.loc[mask, 'intervention_count_norm'] = (
            (group['intervention_count'] - group['intervention_count'].min()) / 
            (group['intervention_count'].max() - group['intervention_count'].min())
        )
    
    return df_norm

______________________________________________________________

In [ ]:
df_plot = df.copy()
df_plot['augmentation'] = df_plot['augmentation'].replace({'rain_torrential': 'rain', 'sunFlare_physic': 'sunflare', 'no_augmentation': 'no_aug'})

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.catplot(
    data=df_plot, x="augmentation", y="path_length",
    hue=, col="environment",
    kind="bar", errorbar="sd", height=4, aspect=1.1, 
)
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from matplotlib.gridspec import GridSpec

In [ ]:
def plot_success_metrics(df, robot, environment):
    """
    Plot binary success metrics (arrival, arrival_safe, false_goal_positive)
    as grouped bar charts by augmentation
    """
    # Filter data
    data = df[(df['robot'] == robot) & (df['environment'] == environment)]
    
    # Group by augmentation and calculate success rates
    success_metrics = data.groupby('augmentation').agg({
        'arrival': 'mean',
        'arrival_safe': 'mean',
        'false_goal_positive': 'mean'
    }).reset_index()
    
    # Create plot
    fig, ax = plt.subplots(figsize=(10, 6))
    
    x = np.arange(len(success_metrics))
    width = 0.25
    
    bars1 = ax.bar(x - width, success_metrics['arrival'] * 100, width, 
                   label='Arrival Rate', color="#69e886", alpha=0.8)
    bars2 = ax.bar(x, success_metrics['arrival_safe'] * 100, width,
                   label='Safe Arrival Rate', color="#57b1d4", alpha=0.8)
    bars3 = ax.bar(x + width, success_metrics['false_goal_positive'] * 100, width,
                   label='False Goal Rate', color="#e7af55", alpha=0.8)
    
    ax.set_xlabel('Augmentation', fontsize=12, fontweight='bold')
    ax.set_ylabel('Percentage (%)', fontsize=12, fontweight='bold')
    ax.set_title(f'Success Metrics: {robot} in {environment}', 
                 fontsize=14, fontweight='bold', pad=20)
    ax.set_xticks(x)
    ax.set_xticklabels(success_metrics['augmentation'], rotation=45, ha='right')
    ax.legend(frameon=True, shadow=True)
    ax.set_ylim(0, 105)
    
    # Add value labels on bars
    for bars in [bars1, bars2, bars3]:
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{height:.1f}%', ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    return fig


_ = plot_success_metrics(df_plot[df_plot['augmentation'] != reference_header], robot="bunker", environment="mist_corridor")

In [ ]:
def plot_performance_metrics(df, robot, environment):
    """
    Plot continuous performance metrics (path_length, avg_velocity, dist_to_goal)
    as box plots by augmentation
    """
    data = df[(df['robot'] == robot) & (df['environment'] == environment)]
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    metrics = [
        ('path_length', 'Path Length', '#9b59b6'),
        ('avg_velocity', 'Average Velocity', '#f39c12'),
        ('dist_to_goal', 'Distance to Goal', '#e67e22')
    ]
    
    for ax, (metric, title, color) in zip(axes, metrics):
        # Create box plot
        augmentations = data['augmentation'].unique()
        plot_data = [data[data['augmentation'] == aug][metric].dropna() 
                     for aug in augmentations]
        
        bp = ax.boxplot(plot_data, labels=augmentations, patch_artist=True,
                        showmeans=True, meanline=True)
        
        # Color the boxes
        for patch in bp['boxes']:
            patch.set_facecolor(color)
            patch.set_alpha(0.6)
        
        # Style median and mean lines
        for median in bp['medians']:
            median.set_color('black')
            median.set_linewidth(2)
        for mean in bp['means']:
            mean.set_color('red')
            mean.set_linewidth(2)
            mean.set_linestyle('--')
        
        ax.set_xlabel('Augmentation', fontsize=11, fontweight='bold')
        ax.set_ylabel(title, fontsize=11, fontweight='bold')
        ax.set_title(title, fontsize=12, fontweight='bold')
        ax.tick_params(axis='x', rotation=45)
        ax.grid(axis='y', alpha=0.3)
    
    plt.suptitle(f'Performance Metrics: {robot} in {environment}', 
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    return fig

**Only for medium / hard setting**

In [ ]:
import pandas as pd

success_rate = (
    df.groupby(["robot", "environment", "augmentation"])["arrival_safe"]
    .mean().reset_index()
)

sns.barplot(
    data=success_rate,
    x="augmentation", y="arrival_safe", hue="robot"
    
)
plt.ylabel("Safe Arrival Rate")
plt.show()


_______________

# IMAGE ANALYSIS HANDLED
**PROCRESS IMAGES WITH FIND_LAST-IMG.PY**

In [ ]:
# conda install -c menpo opencv
# pip install image-similarity-measures
import cv2 as cv2
import image_similarity_measures
from image_similarity_measures.quality_metrics import rmse, psnr

In [ ]:
ref_path = "/home/mae/Documents/GIT/Research/SafeGNM/metrics/medias/reference_last.png"
actual_path = "/home/mae/Documents/GIT/Research/SafeGNM/metrics/medias/actual_last.png"

ref_img = cv2.imread(ref_path)
actual_img = cv2.imread(actual_path)

https://up42.com/blog/image-similarity-measures
RMSE: measures the amount of change per pixel due

In [ ]:
print(f"RMSE ref vs actual : {rmse(ref_img, actual_img)}")
print(f"RMSE ref vs ref : {rmse(ref_img, ref_img)}")

**diffimg formula**
https://github.com/nicolashahn/diffimg?tab=readme-ov-file

The difference is defined by the average % difference between each of the channels (R,G,B,A?) at each pair of pixels Axy, Bxy at the same coordinates in each of the two images (why they need to be the same size), averaged over all pairs of pixels.

For example, compare two 1x1 images A and B (a trivial example, >1 pixels would have another step to find the average of all pixels):

A1,1 = RGB(255,0,0) (pure red)

B1,1 = RGB(100,0,0) (dark red)

((255-100)/255 + (0/0)/255 + (0/0)/255))/3 = (155/255)/3 = 0.202614379

So these two 1x1 images differ by 20.2614379% according to this formula.

In [ ]:
# pip install diffimg
from diffimg import diff

print(f"Diff ref vs actual {diff(ref_path, actual_path)}")
print(f"Diff ref vs ref {diff(ref_path, ref_path)}")

_____________